# Practice - 05 - Auto Loader: incremental runs, rescue, and schema evolution

Prerequisites / execution context: Run 00. Uses AvailableNow for serverless compatibility. Creates isolated paths and tables; no cloud notification setup required for the directory-listing exercise.

These are original teaching examples, not official exam questions. Read the linked documentation before executing. Databricks-specific cells require Databricks; opening an IPYNB in local Jupyter does not provide a Databricks runtime.

Official Databricks documentation: [R10: Auto Loader](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader); [R11: Auto Loader schema inference and evolution](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/schema); [R12: Auto Loader file detection modes](https://docs.databricks.com/aws/en/ingestion/cloud-object-storage/auto-loader/file-detection-modes); [R30: Streaming checkpoints](https://docs.databricks.com/aws/en/structured-streaming/checkpoints); [R04: Serverless compute limitations](https://docs.databricks.com/aws/en/compute/serverless/limitations).


**Try first:** complete each YOUR CODE / YOUR ANSWER cell. Only then run the solution-display cell and click Reveal. Solutions are feedback text, never automatically executed. Cells marked check compare your own implementation. Work top to bottom; do not use Run All while placeholders remain.

**Isolation:** the default prefix in this edition is dea_riccardo_practice. Run the practice setup before later practice notebooks. For opt-in cells, complete the code before enabling the flag.


In [ ]:
# Use the SAME values throughout the kit. Select an existing writable UC catalog.
CATALOG = "workspace"
PREFIX = "dea_riccardo_practice"
import re
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", CATALOG)
assert re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", PREFIX)
BRONZE, SILVER, GOLD = [f"{CATALOG}.{PREFIX}_{x}" for x in ("bronze", "silver", "gold")]
VOLUME = f"/Volumes/{CATALOG}/{PREFIX}_bronze/landing"
from pyspark.sql import functions as F, Window
spark.conf.set("spark.sql.session.timeZone", "UTC")


In [ ]:
# Run once. This helper displays a solution; it never executes solution code.
import base64 as _practice_base64

def show_solution(encoded_html):
    panel = _practice_base64.b64decode(encoded_html).decode("utf-8")
    try:
        renderer = displayHTML  # Databricks
    except NameError:
        from IPython.display import display, HTML
        display(HTML(panel))
    else:
        renderer(panel)


## Coding task 1: Implement an incremental ingestion function

Create isolated source, schema_path, checkpoint and T variables. Write a.json with string id=1 and amount=10. Implement ingest(): cloudFiles JSON source, rescue schema policy, _rescued_data, persistent state, AvailableNow Delta table sink, await completion, return query. Invoke twice and compare counts.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE
import uuid
run_id = uuid.uuid4().hex[:10]
base = f"{VOLUME}/auto_{run_id}"
source = f"{base}/input"
schema_path = f"{base}/schema"
checkpoint = f"{base}/checkpoint"
T = f"{BRONZE}.auto_{run_id}"
dbutils.fs.put(f"{source}/a.json", '{"id":"1","amount":"10"}', True)

# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# CHECK YOUR IMPLEMENTATION - run only after completing the task above.
assert spark.table(T).count() == 1
print("Checks passed")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDEuIEluZ2VzdCBpbml0aWFsIEpTT04gd2l0aCBhIHBlcnNpc3RlbnQgY2hlY2twb2ludApLZWVwIHRoaXMgcnVuX2lkIGFuZCBib3RoIHN0YXRlIGxvY2F0aW9ucyB1bmNoYW5nZWQgZm9yIGFsbCBjZWxscyBpbiB0aGlzIGV4cGVyaW1lbnQuCgoKVGhlIHJlZmVyZW5jZSBpbXBsZW1lbnRhdGlvbiBpcyBzaG93biBuZXh0LiBDb21wYXJlIHlvdXIgZGVzaWduIGFuZCByZXN1bHRzIGJlZm9yZSBjb3B5aW5nIGl0LjwvZGl2PjwvZGV0YWlscz4=')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAxIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPmltcG9ydCB1dWlkCnJ1bl9pZD11dWlkLnV1aWQ0KCkuaGV4WzoxMF0KYmFzZT1mJnF1b3Q7e1ZPTFVNRX0vYXV0b197cnVuX2lkfSZxdW90Owpzb3VyY2U9ZiZxdW90O3tiYXNlfS9pbnB1dCZxdW90Ozsgc2NoZW1hX3BhdGg9ZiZxdW90O3tiYXNlfS9zY2hlbWEmcXVvdDs7IGNoZWNrcG9pbnQ9ZiZxdW90O3tiYXNlfS9jaGVja3BvaW50JnF1b3Q7ClQ9ZiZxdW90O3tCUk9OWkV9LmF1dG9fe3J1bl9pZH0mcXVvdDsKZGJ1dGlscy5mcy5wdXQoZiZxdW90O3tzb3VyY2V9L2EuanNvbiZxdW90OywmI3gyNzt7JnF1b3Q7aWQmcXVvdDs6JnF1b3Q7MSZxdW90OywmcXVvdDthbW91bnQmcXVvdDs6JnF1b3Q7MTAmcXVvdDt9JiN4Mjc7LFRydWUpCmRlZiBpbmdlc3QoKToKICAgIGRmPShzcGFyay5yZWFkU3RyZWFtLmZvcm1hdCgmcXVvdDtjbG91ZEZpbGVzJnF1b3Q7KS5vcHRpb24oJnF1b3Q7Y2xvdWRGaWxlcy5mb3JtYXQmcXVvdDssJnF1b3Q7anNvbiZxdW90OykKICAgICAgICAub3B0aW9uKCZxdW90O2Nsb3VkRmlsZXMuc2NoZW1hTG9jYXRpb24mcXVvdDssc2NoZW1hX3BhdGgpCiAgICAgICAgLm9wdGlvbigmcXVvdDtjbG91ZEZpbGVzLnNjaGVtYUV2b2x1dGlvbk1vZGUmcXVvdDssJnF1b3Q7cmVzY3VlJnF1b3Q7KQogICAgICAgIC5vcHRpb24oJnF1b3Q7cmVzY3VlZERhdGFDb2x1bW4mcXVvdDssJnF1b3Q7X3Jlc2N1ZWRfZGF0YSZxdW90OykubG9hZChzb3VyY2UpKQogICAgcT0oZGYud3JpdGVTdHJlYW0ub3B0aW9uKCZxdW90O2NoZWNrcG9pbnRMb2NhdGlvbiZxdW90OyxjaGVja3BvaW50KQogICAgICAgLnRyaWdnZXIoYXZhaWxhYmxlTm93PVRydWUpLnRvVGFibGUoVCkpCiAgICBxLmF3YWl0VGVybWluYXRpb24oKQogICAgcmV0dXJuIHEKcT1pbmdlc3QoKQphc3NlcnQgc3BhcmsudGFibGUoVCkuY291bnQoKT09MQpxPWluZ2VzdCgpCmFzc2VydCBzcGFyay50YWJsZShUKS5jb3VudCgpPT0xCjwvY29kZT48L3ByZT48L2RldGFpbHM+')


## Coding task 2: Handle an unexpected JSON field

Add b.json with id=2,amount=20,currency=EUR (all strings). Run ingest(), inspect the target schema and rescued rows, and inspect query progress. Name the target DataFrame result.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE


# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# CHECK YOUR IMPLEMENTATION - run only after completing the task above.
assert result.count() == 2
assert "currency" not in result.columns
assert result.filter("_rescued_data IS NOT NULL").count() == 1
print("Checks passed")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDIuIEFkZCBhIGNvbHVtbiBhbmQgaW5zcGVjdCByZXNjdWVkIGRhdGEKQmVjYXVzZSByZXNjdWUgbW9kZSB3YXMgc2VsZWN0ZWQsIHRoZSBuZXcgY3VycmVuY3kgZmllbGQgc2hvdWxkIGJlIGNhcHR1cmVkIGluIF9yZXNjdWVkX2RhdGEsIG5vdCBhZGRlZCB0byB0aGUgaW5mZXJyZWQgc2NoZW1hLiBTb3VyY2Utc2NoZW1hIGV2b2x1dGlvbiBhbmQgdGFyZ2V0LXRhYmxlIHNjaGVtYSBldm9sdXRpb24gYXJlIHNlcGFyYXRlIGNvbnRyb2xzLgoKClRoZSByZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gaXMgc2hvd24gbmV4dC4gQ29tcGFyZSB5b3VyIGRlc2lnbiBhbmQgcmVzdWx0cyBiZWZvcmUgY29weWluZyBpdC48L2Rpdj48L2RldGFpbHM+')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAyIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPmRidXRpbHMuZnMucHV0KGYmcXVvdDt7c291cmNlfS9iLmpzb24mcXVvdDssJiN4Mjc7eyZxdW90O2lkJnF1b3Q7OiZxdW90OzImcXVvdDssJnF1b3Q7YW1vdW50JnF1b3Q7OiZxdW90OzIwJnF1b3Q7LCZxdW90O2N1cnJlbmN5JnF1b3Q7OiZxdW90O0VVUiZxdW90O30mI3gyNzssVHJ1ZSkKcT1pbmdlc3QoKQpyZXN1bHQ9c3BhcmsudGFibGUoVCkKYXNzZXJ0IHJlc3VsdC5jb3VudCgpPT0yCmFzc2VydCAmcXVvdDtjdXJyZW5jeSZxdW90OyBub3QgaW4gcmVzdWx0LmNvbHVtbnMKYXNzZXJ0IHJlc3VsdC5maWx0ZXIoJnF1b3Q7X3Jlc2N1ZWRfZGF0YSBJUyBOT1QgTlVMTCZxdW90OykuY291bnQoKT09MQpyZXN1bHQuc2hvdyh0cnVuY2F0ZT1GYWxzZSkKcHJpbnQocS5sYXN0UHJvZ3Jlc3MpCjwvY29kZT48L3ByZT48L2RldGFpbHM+')


## Coding task 3: Optional: evolve the schema

Use entirely separate input/schema/checkpoint/target locations. Infer an id-only schema, then add currency. Implement addNewColumns plus target mergeSchema. Predict the first schema-change failure and plan the restart. Do not delete checkpoints or catch every exception as a retry.

Use the official references in this notebook header. Prior variables remain available only after you implement preceding tasks.


In [ ]:
# YOUR CODE HERE
RUN_EVOLUTION_EXPERIMENT = False
# Keep disabled until you have designed the isolated experiment.

# Replace the following line with your implementation.
raise NotImplementedError("Complete this exercise before continuing")


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAzIGV4cGxhbmF0aW9uPC9zdW1tYXJ5PjxkaXYgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPiMjIDMuIE9wdGlvbmFsIGNvbnRyb2xsZWQgc2NoZW1hLWV2b2x1dGlvbiBleHBlcmltZW50ClVzZSBhIHNlcGFyYXRlIG5ldyBwYXRoLCBzY2hlbWEgZGlyZWN0b3J5LCBjaGVja3BvaW50IGFuZCB0YXJnZXQuIEZpcnN0IGluZ2VzdCBhIGZpbGUgY29udGFpbmluZyBvbmx5IGlkLiBUaGVuIGFkZCBhIGZpbGUgY29udGFpbmluZyBpZCBhbmQgY3VycmVuY3kuIENvbmZpZ3VyZSBhZGROZXdDb2x1bW5zIHdpdGhvdXQgc3VwcGx5aW5nIGEgZml4ZWQgc2NoZW1hOyBlbmFibGUgbWVyZ2VTY2hlbWEgb24gdGhlIHNpbmsuIEV4cGVjdCB0aGUgZmlyc3QgcnVuIGFmdGVyIHRoZSBuZXcgZmllbGQgdG8gZmFpbCBhZnRlciB1cGRhdGluZyB0aGUgaW5mZXJyZWQgc2NoZW1hLiBSZXN0YXJ0IHdpdGggdGhlIFNBTUUgc3RhdGUgbG9jYXRpb25zOyBleHBlY3QgY3VycmVuY3kgdG8gYmVjb21lIGEgY29sdW1uLiBEbyBub3QgdHVybiBhbiBhcmJpdHJhcnkgcHJvZHVjdGlvbiBleGNlcHRpb24gaW50byBhbiB1bmNvbmRpdGlvbmFsIHJldHJ5IGxvb3AuIE9mZmljaWFsIERhdGFicmlja3MgZG9jdW1lbnRhdGlvbjogPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9pbmdlc3Rpb24vY2xvdWQtb2JqZWN0LXN0b3JhZ2UvYXV0by1sb2FkZXIvc2NoZW1hIiB0YXJnZXQ9Il9ibGFuayIgcmVsPSJub29wZW5lciI+UjExOiBBdXRvIExvYWRlciBzY2hlbWEgaW5mZXJlbmNlIGFuZCBldm9sdXRpb248L2E+LgoKClRoZSByZWZlcmVuY2UgaW1wbGVtZW50YXRpb24gaXMgc2hvd24gbmV4dC4gQ29tcGFyZSB5b3VyIGRlc2lnbiBhbmQgcmVzdWx0cyBiZWZvcmUgY29weWluZyBpdC48L2Rpdj48L2RldGFpbHM+')


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgdGFzayAzIHJlZmVyZW5jZSBjb2RlPC9zdW1tYXJ5PjxwcmUgc3R5bGU9IndoaXRlLXNwYWNlOnByZS13cmFwO292ZXJmbG93LXdyYXA6YW55d2hlcmUiPjxjb2RlPlJVTl9FVk9MVVRJT05fRVhQRVJJTUVOVD1GYWxzZQppZiBSVU5fRVZPTFVUSU9OX0VYUEVSSU1FTlQ6CiAgICBldm89ZiZxdW90O3tWT0xVTUV9L2V2b197dXVpZC51dWlkNCgpLmhleFs6MTBdfSZxdW90OwogICAgZXZvX3RhYmxlPWYmcXVvdDt7QlJPTlpFfS5ldm9fe3V1aWQudXVpZDQoKS5oZXhbOjEwXX0mcXVvdDsKICAgIGRidXRpbHMuZnMucHV0KGYmcXVvdDt7ZXZvfS9pbnB1dC9hLmpzb24mcXVvdDssJiN4Mjc7eyZxdW90O2lkJnF1b3Q7OiZxdW90OzEmcXVvdDt9JiN4Mjc7LFRydWUpCiAgICBkZWYgZXZvbHZlKCk6CiAgICAgICAgZGY9KHNwYXJrLnJlYWRTdHJlYW0uZm9ybWF0KCZxdW90O2Nsb3VkRmlsZXMmcXVvdDspLm9wdGlvbigmcXVvdDtjbG91ZEZpbGVzLmZvcm1hdCZxdW90OywmcXVvdDtqc29uJnF1b3Q7KQogICAgICAgICAgICAub3B0aW9uKCZxdW90O2Nsb3VkRmlsZXMuc2NoZW1hTG9jYXRpb24mcXVvdDssZiZxdW90O3tldm99L3NjaGVtYSZxdW90OykKICAgICAgICAgICAgLm9wdGlvbigmcXVvdDtjbG91ZEZpbGVzLnNjaGVtYUV2b2x1dGlvbk1vZGUmcXVvdDssJnF1b3Q7YWRkTmV3Q29sdW1ucyZxdW90OykubG9hZChmJnF1b3Q7e2V2b30vaW5wdXQmcXVvdDspKQogICAgICAgIHE9KGRmLndyaXRlU3RyZWFtLm9wdGlvbigmcXVvdDtjaGVja3BvaW50TG9jYXRpb24mcXVvdDssZiZxdW90O3tldm99L2NoZWNrcG9pbnQmcXVvdDspCiAgICAgICAgICAgIC5vcHRpb24oJnF1b3Q7bWVyZ2VTY2hlbWEmcXVvdDssJnF1b3Q7dHJ1ZSZxdW90OykudHJpZ2dlcihhdmFpbGFibGVOb3c9VHJ1ZSkudG9UYWJsZShldm9fdGFibGUpKQogICAgICAgIHEuYXdhaXRUZXJtaW5hdGlvbigpCiAgICBldm9sdmUoKQogICAgZGJ1dGlscy5mcy5wdXQoZiZxdW90O3tldm99L2lucHV0L2IuanNvbiZxdW90OywmI3gyNzt7JnF1b3Q7aWQmcXVvdDs6JnF1b3Q7MiZxdW90OywmcXVvdDtjdXJyZW5jeSZxdW90OzomcXVvdDtFVVImcXVvdDt9JiN4Mjc7LFRydWUpCiAgICAjIEV4cGVjdGVkIHRvIGZhaWwgb25jZSB3aGVuIHRoZSBuZXcgY29sdW1uIGlzIGRpc2NvdmVyZWQuCiAgICAjIEFmdGVyIGluc3BlY3RpbmcgdGhhdCBzcGVjaWZpYyBlcnJvciwgcnVuIGV2b2x2ZSgpIGluIGEgTkVXIGNlbGwuCiAgICBldm9sdmUoKQo8L2NvZGU+PC9wcmU+PC9kZXRhaWxzPg==')


## Written / design task 1

Differentiate schema state from checkpoint progress. What happens if you reset the checkpoint while preserving the target? Compare file discovery modes with trigger choice and ordering guarantees.


**YOUR ANSWER**

_Write your prediction, design or explanation here before opening the solution._


In [ ]:
# Optional: run this cell, then click "Reveal". This only displays the answer.
show_solution('PGRldGFpbHMgc3R5bGU9ImJvcmRlcjoxcHggc29saWQgIzgxOTVhNTtib3JkZXItcmFkaXVzOjdweDtwYWRkaW5nOjEycHgiPjxzdW1tYXJ5IHN0eWxlPSJjdXJzb3I6cG9pbnRlcjtmb250LXdlaWdodDpib2xkIj5SZXZlYWwgd3JpdHRlbiB0YXNrIDEgZmVlZGJhY2s8L3N1bW1hcnk+PGRpdiBzdHlsZT0id2hpdGUtc3BhY2U6cHJlLXdyYXA7b3ZlcmZsb3ctd3JhcDphbnl3aGVyZSI+IyMgRXhwbGFpbiBiZWZvcmUgbW92aW5nIG9uClRoZSBzY2hlbWEgZGlyZWN0b3J5IHRyYWNrcyB0aGUgcmVhZGVyIHNjaGVtYTsgdGhlIGNoZWNrcG9pbnQgdHJhY2tzIHByb2dyZXNzLiBEZWxldGluZyBvbmx5IHRoZSBjaGVja3BvaW50IGNhbiByZXBsYXkgb2xkIGZpbGVzIGludG8gYW4gZXhpc3RpbmcgdGFyZ2V0LiBGaWxlIGRpc2NvdmVyeSBtb2RlIGlzIHNlcGFyYXRlIGZyb20gQXZhaWxhYmxlTm93LCB3aGljaCBjb250cm9scyBxdWVyeSBleGVjdXRpb24uIEZvciBwcm9kdWN0aW9uLCBldmFsdWF0ZSBtYW5hZ2VkIGZpbGUgZXZlbnRzIG9uIGEgc3VwcG9ydGVkIGV4dGVybmFsIGxvY2F0aW9uLiBBdXRvIExvYWRlciBkb2VzIG5vdCBwcm9taXNlIGZpbGUgb3JkZXIuIE9mZmljaWFsIERhdGFicmlja3MgZG9jdW1lbnRhdGlvbjogPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9pbmdlc3Rpb24vY2xvdWQtb2JqZWN0LXN0b3JhZ2UvYXV0by1sb2FkZXIvZmlsZS1kZXRlY3Rpb24tbW9kZXMiIHRhcmdldD0iX2JsYW5rIiByZWw9Im5vb3BlbmVyIj5SMTI6IEF1dG8gTG9hZGVyIGZpbGUgZGV0ZWN0aW9uIG1vZGVzPC9hPjsgPGEgaHJlZj0iaHR0cHM6Ly9kb2NzLmRhdGFicmlja3MuY29tL2F3cy9lbi9zdHJ1Y3R1cmVkLXN0cmVhbWluZy9jaGVja3BvaW50cyIgdGFyZ2V0PSJfYmxhbmsiIHJlbD0ibm9vcGVuZXIiPlIzMDogU3RyZWFtaW5nIGNoZWNrcG9pbnRzPC9hPi4KPC9kaXY+PC9kZXRhaWxzPg==')
